#1.0 Exploratory Data Analysis with Big Data

##1.1 Loading and Inspecting the Data

In [0]:
# ──────────────────────────────────────────────────────────────────────────────
# NFL Big Data Bowl 2027 — Data Ingestion
# Loads 2 Unity Catalog tables and 7 CSV files from a UC volume as Spark
# DataFrames for downstream analysis.
# ──────────────────────────────────────────────────────────────────────────────

VOLUME_PATH = "/Volumes/nfl2027/nfl2027/nfl_2027"


# ── Helper: load a CSV from the Unity Catalogue volume with header + schema inference ──
def load_volume_csv(filename, volume_path=VOLUME_PATH):
    """Read a CSV file from the Unity Catalogue volume as a Spark DataFrame."""
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{volume_path}/{filename}")
    )


# ── 1. Unity Catalog Tables (2) ──
# Pre-registered Delta tables — loaded directly via spark.table()
combine_results_df       = spark.table("nfl2027.nfl2027.combine_results")
game_tracking_2025_df    = spark.table("nfl2027.nfl2027.game_tracking_2025")

# ── 2. Volume CSV Files (7) ──
# Raw CSVs stored in the nfl_2027 Unity Catalog volume
combine_tracking_df         = load_volume_csv("combine_tracking.csv")
game_tracking_2023_df       = load_volume_csv("game_tracking_2023.csv")
game_tracking_2024_df       = load_volume_csv("game_tracking_2024.csv")
games_df                    = load_volume_csv("games.csv")
player_career_successes_df  = load_volume_csv("player_career_successes.csv")
player_play_df              = load_volume_csv("player_play.csv")
players_df                  = load_volume_csv("players.csv")

print("✅ All 9 DataFrames loaded successfully.")

In [0]:
# ── Quick Verification: schema + row counts for all 9 DataFrames ──
# Lightweight check — schema is free; counts trigger a scan.

dataframes = {
    # Unity Catalog tables
    "combine_results_df"      : combine_results_df,
    "game_tracking_2025_df"   : game_tracking_2025_df,
    # Volume CSV files
    "combine_tracking_df"         : combine_tracking_df,
    "game_tracking_2023_df"       : game_tracking_2023_df,
    "game_tracking_2024_df"       : game_tracking_2024_df,
    "games_df"                    : games_df,
    "player_career_successes_df"  : player_career_successes_df,
    "player_play_df"              : player_play_df,
    "players_df"                  : players_df,
}

for name, df in dataframes.items():
    print(f"{name}: {len(df.columns)} cols")
    df.printSchema()

In [0]:
for name, df in dataframes.items():
    print(f"{name}: {df.count()} rows × {len(df.columns)} cols")

In [0]:
#checking for descriptive of 40 yard dash
dataframes["combine_results_df"].describe("forty").show()

In [0]:

#converting forty column from data type string to double

from pyspark.sql.functions import col, expr
# Cast the 'forty' column to double, using try_cast to handle 'NA' values as NULL
combine_results_df = combine_results_df.withColumn("forty", expr("try_cast(forty as double)"))
dataframes["combine_results_df"] = combine_results_df

print("✅ 'forty' column cast to double.")
combine_results_df.printSchema()
combine_results_df.select("forty").show(5)

In [0]:
#descriptives of 40 yard dash after fixing datatypes
dataframes["combine_results_df"].describe("forty").show()

In [0]:
#selecting a specific column within a dataframe and checking for distinct values
dataframes["combine_tracking_df"].select("drill_name").distinct().count()

In [0]:
dataframes["combine_tracking_df"].select("drill_name").distinct().show()

## 1.2 Cleaning and Transforming the data

In [0]:
#checking for null values
from pyspark.sql.functions import col, count, when, sum as _sum

for name, df in dataframes.items():
    null_counts = df.select(
        [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
    ).collect()[0].asDict()
    total_nulls = sum(null_counts.values())
    print(f"{name}: {total_nulls} total nulls across {len(df.columns)} columns")
    print()